In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy.special import expit
from IPython.display import display

In [2]:
TASK = Path.cwd().parent
ROOT = TASK.parent
INPUT, OUTPUT = TASK/'input', TASK/'output'
CONFIG = json.loads((ROOT/'config.json').read_text())
EPS, Y, SEED = CONFIG['eps'], CONFIG['last_year'], CONFIG['seed']
TAG = 'se_hurdle_s'
N = CONFIG['n_sim']
rng = np.random.default_rng(SEED)
DEST = OUTPUT/TAG
DEST.mkdir(exist_ok=True)

In [3]:
params = pd.read_csv(INPUT/TAG/'parameters.csv', dtype={'stage':str})
rho = pd.read_csv(INPUT/TAG/'memory.csv').rho.iloc[0]
panel = pd.read_csv(INPUT/'data/panel_complete.csv', index_col='dblp_id')
lookup = params.set_index(['stage','equation'])

In [4]:
# original SE initialization; all later years generated
sim = np.zeros((Y+1, N))
sim[0] = rng.choice(panel['0'].to_numpy(), N, replace=True)
numerator, denominator = np.zeros(N), 0.
for t in range(Y):
    stage = '0' if t==0 else '1-4' if t<=4 else '5-7' if t<=7 else '8-20'
    previous = sim[t]
    history = numerator/denominator if denominator>0 else np.zeros(N)
    x = np.log1p(previous)
    activity = lookup.loc[(stage,'activity')]
    eta = activity.intercept + activity.x_prev*x + activity.prev_active*(previous>0) + activity.history*history
    active = rng.random(N) < expit(np.clip(eta, -35, 35))
    positive = lookup.loc[(stage,'positive')]
    mean = positive.intercept + positive.x_prev*x[active] + positive.restart*(previous[active]<=0) + positive.history*history[active]
    # legacy normal log-intensity residuals and numerical guard
    sim[t+1,active] = np.exp(np.clip(mean+rng.normal(0, positive.sigma, active.sum()), -30, 30))
    numerator = rho*numerator + x
    denominator = rho*denominator + 1
np.save(DEST/'trajectories.npy', sim, allow_pickle=False)
pd.DataFrame([{'model':TAG, 'seed':SEED, 'n_sim':N, 'n_years':Y+1, 'generated_from_age':1,
               'orientation':'age x scholar', 'minimum':sim.min(), 'maximum':sim.max(),
               'finite':np.isfinite(sim).all()}]).to_csv(DEST/'metadata.csv', index=False)
display(pd.read_csv(DEST/'metadata.csv'))

,model,seed,n_sim,n_years,generated_from_age,orientation,minimum,maximum,finite
0,se_hurdle_s,63,10000,21,1,age x scholar,0.0,222.097509,True
